# Forced and Damped Harmonic Oscillator

Explore the motion of a damped oscillator driven by a sinusoidal force. Adjust the driving angular frequency and damping coefficient to compare the total motion with its transient and steady-state parts.

This notebook uses `ipywidgets` controls and standard inline Matplotlib output, following the same pattern as the previous oscillator notebooks.

## Model

The equation of motion is

$$
\frac{d^2x}{dt^2} + 2\gamma\frac{dx}{dt} + \omega_0^2x = \frac{F_0}{m}\cos(\omega_f t).
$$

The natural angular frequency is fixed at $\omega_0=1$, and the force-to-mass ratio is fixed at $F_0/m=1$. The solution is the sum of a transient term, which decays with time, and a steady-state term, which oscillates at the driving frequency.

The initial position and velocity are fixed at $x(0)=5$ and $v(0)=0$. The sliders change the driving frequency $\omega_f$ and damping coefficient $\gamma$.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

print('NumPy:', np.__version__)
print('Matplotlib:', plt.matplotlib.__version__)
print('ipywidgets:', widgets.__version__)
print('Matplotlib backend:', plt.get_backend())

NumPy: 2.4.6
Matplotlib: 3.10.9
ipywidgets: 8.1.9
Matplotlib backend: module://matplotlib_inline.backend_inline


In [2]:
# Fixed model parameters
x0 = 5.0
v0 = 0.0
F0_over_m = 1.0
w0 = 1.0
t = np.linspace(0.0, 100.0, 5000)

# Interactive controls
sfreq = widgets.FloatSlider(
    value=0.5, min=0.1, max=3.0, step=0.01,
    description='Driving frequency ωf', continuous_update=False,
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='500px')
)
sgam = widgets.FloatSlider(
    value=0.1, min=0.01, max=0.999, step=0.001,
    description='Damping coefficient γ', continuous_update=False,
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='500px')
)

def forced_oscillator(wf, gamma):
    # Underdamped transient response: gamma < w0
    wd = np.sqrt(w0**2 - gamma**2)

    # Steady-state amplitude and phase lag. arctan2 handles all quadrants.
    amp = F0_over_m / np.sqrt((w0**2 - wf**2)**2 + (2*gamma*wf)**2)
    delta = np.arctan2(2*gamma*wf, w0**2 - wf**2)
    steady = amp * np.cos(wf*t - delta)

    # Choose transient coefficients to satisfy the specified initial conditions.
    C = x0 - amp*np.cos(delta)
    D = (v0 + gamma*C - amp*wf*np.sin(delta)) / wd
    transient = np.exp(-gamma*t) * (C*np.cos(wd*t) + D*np.sin(wd*t))

    total = transient + steady
    return total, transient, steady, amp

def plot_forced_oscillator(wf, gamma):
    total, transient, steady, amp = forced_oscillator(wf, gamma)

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(t, total, color='red', linewidth=1.8, label='Total motion')
    ax.plot(t, transient, color='blue', linewidth=1.0, label='Transient part')
    ax.plot(t, steady, color='green', linewidth=1.0, label='Steady-state part')
    ax.set_xlim(0, 100)
    ax.set_xlabel('Time')
    ax.set_ylabel('Position, x')
    ax.set_title(f'Forced and Damped Oscillator: ωf = {wf:.2f}, γ = {gamma:.3f}')
    ax.grid(True, alpha=0.3)
    ax.legend()
    fig.tight_layout()
    plt.show()
    plt.close(fig)

plot_output = widgets.interactive_output(
    plot_forced_oscillator,
    {'wf': sfreq, 'gamma': sgam}
)

reset_button = widgets.Button(description='Reset')
def reset_values(button):
    sfreq.value = 0.5
    sgam.value = 0.1
reset_button.on_click(reset_values)

display(widgets.VBox([sfreq, sgam, reset_button]), plot_output)

Output()

## Questions to investigate

1. What happens to the steady-state amplitude as the driving frequency approaches the natural frequency?
2. How does increasing the damping coefficient change the steady-state amplitude near resonance?
3. How does the transient part change as time increases?
4. After the transient has mostly decayed, which part of the solution remains?

## Notes

The transient term is calculated so that the total solution satisfies the initial position and velocity. The steady-state amplitude is not necessarily the largest amplitude of the total motion because the transient and steady-state terms can add or cancel. The vertical axis is allowed to autoscale so large amplitudes near resonance are not clipped.